# Exercises - Lesson 1.2: Math Foundations 2 - How Models Learn

In [ ]:
!pip install -q numpy torch

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Skill 1 - Cross-entropy loss

In [ ]:
def softmax(x):
    s = x - np.max(x)
    e = np.exp(s)
    return e / e.sum()

def cross_entropy(logits, true_idx):
    probs = softmax(logits)
    return -np.log(probs[true_idx] + 1e-12)

logits = np.array([2.0, 1.0, 0.1])
loss = cross_entropy(logits, true_idx=0)
print(f'probs = {softmax(logits).round(3)}')
print(f'loss  = {loss:.3f}')

## Skill 2 - Gradient (manual)

In [ ]:
def loss(w): return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

w = 0.0
print(f'L({w}) = {loss(w)}, grad = {grad(w)}')

## Skill 3 - Gradient descent

In [ ]:
def descend(w_init, lr, steps=30):
    w = w_init
    for _ in range(steps):
        w = w - lr * grad(w)
    return w

for lr in [0.01, 0.1, 1.5]:
    final_w = descend(0.0, lr)
    print(f'LR={lr:4.2f}  final w={final_w:.2e}')

## Skill 4 - Backpropagation via PyTorch autograd

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 4), nn.ReLU(),
    nn.Linear(4, 2), nn.ReLU(),
    nn.Linear(2, 1),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
loss_fn = nn.MSELoss()
x = torch.tensor([[0.5, 0.3]])
y = torch.tensor([[1.0]])

for step in range(200):
    optimizer.zero_grad()
    y_hat = model(x)
    l = loss_fn(y_hat, y)
    l.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

print(f'final loss = {l.item():.4f}')
print(f'final y_hat = {y_hat.item():.3f}')

## Full production training loop

In [ ]:
model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

for step in range(100):
    x = torch.randn(16, 10)
    y = torch.randint(0, 2, (16,))
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
print(f'final loss = {loss.item():.4f}')